In [1]:
# ─── Environment Setup (do not edit) ────────────────────────────────────────
import os, sys
from pathlib import Path


def _detect_platform():
    try:
        import google.colab  # noqa: F401

        return "colab", False
    except ImportError:
        pass
    if Path("/workspace").exists() and os.environ.get("VAST_CONTAINERLABEL"):
        return "vastai", False
    if Path("/workspace").exists():
        return "vastai", True
    if sys.platform == "win32":
        return "windows", False
    if sys.platform == "darwin":
        return "mac", False
    return None, True


PLATFORM, _uncertain = _detect_platform()

if PLATFORM == "colab":
    from google.colab import drive

    drive.mount("/content/drive")

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

if PLATFORM == "colab":
    PROJECT_ROOT = Path("/content/drive/MyDrive/Thesis_Final/fake-news-detection")
else:
    PROJECT_ROOT = _nb_path.parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {
    "colab": PROJECT_ROOT / ".env.colab",
    "vastai": PROJECT_ROOT / ".env.vastai",
    "windows": PROJECT_ROOT / ".env.windows",
    "mac": PROJECT_ROOT / ".env.mac",
}

if PLATFORM is None:
    print("⚠️  WARNING: Could not detect platform. Falling back to .env (local).")
    _env_file = PROJECT_ROOT / ".env"
elif _uncertain:
    print(
        f"⚠️  WARNING: Detected /workspace but VAST_CONTAINERLABEL is not set. Assuming Vast.ai."
    )
    _env_file = _env_map["vastai"]
else:
    _env_file = _env_map[PLATFORM]

from dotenv import load_dotenv

if not _env_file.exists():
    _fallback = PROJECT_ROOT / ".env"
    print(f"⚠️  WARNING: Expected env file not found: {_env_file}")
    if _fallback.exists():
        print(f"   Falling back to: {_fallback}")
        _env_file = _fallback
    else:
        raise FileNotFoundError(
            f"No .env file found. Copy the correct example file:\n"
            f"  cp .env.{PLATFORM or 'mac'}.example .env.{PLATFORM or 'mac'}"
        )

load_dotenv(_env_file, override=True)

from src.utils.env_utils import get_data_root

DATA_ROOT = get_data_root()

print(f"✅ Platform : {PLATFORM or 'unknown (local fallback)'}")
print(f"✅ Env file : {_env_file}")
print(f"✅ DATA_ROOT: {DATA_ROOT}")
print(f"{'✅' if DATA_ROOT.exists() else '❌'} Path exists: {DATA_ROOT.exists()}")
# ─────────────────────────────────────────────────────────────────────────────

✅ Platform : mac
✅ Env file : /Users/haila/My File/projects/fake-new-detection/.env.mac
✅ DATA_ROOT: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis
✅ Path exists: True


# ViFactCheck Text Classifier — Stage 3.9

Fine-tunes `vinai/phobert-base-v2` as a Vietnamese fake-news **text-only** classifier on the ViFactCheck
dataset, before the final multimodal Stage 4 fusion. Serves as a strong standalone text baseline and
exports a frozen checkpoint that Stage 4 can optionally load as its text encoder.

**Data sources (priority order):**

1. HuggingFace `tranthaihoa/vifactcheck` — native claim + evidence pairs with labels
2. Fallback: `data/json/labeled_nei_as_real/news_data_vifactcheck_{split}_labeled.json`

**Label strategy (`nei_as_real` — binary):**

-   Supported (0) → Real (0), NEI (2) → Real (0), Refuted (1) → Fake (1)

```
Input:  tranthaihoa/vifactcheck (HF)  OR  labeled_nei_as_real/*.json
Output: training/checkpoints_vifactcheck/<run>/best_model.pth + tokenizer/
        training/checkpoints_vifactcheck/<run>/checkpoint_manifest.json  ← Stage 4 handoff
```


In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "pipeline" else Path.cwd()

try:
    from dotenv import load_dotenv

    load_dotenv(PROJECT_ROOT / ".env", override=False)
except ImportError:
    pass

DATA_ROOT = (
    Path(os.environ["DATA_ROOT"]) if os.environ.get("DATA_ROOT") else PROJECT_ROOT
)

CONFIG = {
    "paths": {
        "labeled_json_dir": DATA_ROOT / "data" / "json",
        "checkpoint_root": DATA_ROOT / "training" / "checkpoints_vifactcheck",
        # Stage 4 integration: PhoBERT per-article features saved here
        "stage39_features_dir": DATA_ROOT / "training" / "stage39_features",
        "mlflow_dir": DATA_ROOT / "mlruns",
    },
    "data": {
        # "huggingface" = load from tranthaihoa/vifactcheck (recommended)
        # "labeled_json" = fallback to labeled_nei_as_real JSON files
        "source": "huggingface",
        "hf_dataset": "tranthaihoa/vifactcheck",
        # None = auto-detect from dataset columns; or set manually
        # Paper columns: Statement (claim), Evidence (gold evidence), Context (full article)
        # Evidence mode (Statement+Evidence) outperforms Context mode per paper §4.3
        "hf_text_field": None,  # auto → 'Statement'
        "hf_context_field": None,  # auto → 'Evidence'  (gold evidence = better mode)
        "hf_label_field": None,  # auto → 'labels'
        # for labeled_json fallback
        "label_variant": "nei_as_real",
        # nei_as_real: Support(0)->0, NEI(2)->0, Refute(1)->1  (6:2:2 ratio becomes ~2:1)
        # binary_exclude_nei: Support(0)->0, Refute(1)->1, NEI->skip
        "label_strategy": "nei_as_real",
        "max_length": 256,
        "num_classes": 2,
        # Word segmentation (paper §4.2: PhoBERT requires VnCoreNLP segmentation)
        # Set True to use underthesea word segmentation (pip install underthesea)
        # False = skip segmentation (still works, ~5% lower F1 expected)
        "word_segment": False,
    },
    "model": {
        "backbone": "vinai/phobert-base-v2",
        # Paper §4.2: dropout=0.3 for PhoBERT on ViFactCheck
        "dropout": 0.3,
        "num_classes": 2,
    },
    "training": {
        "batch_size": 16,  # paper §4.2
        "max_epochs": 15,  # extended from paper's 10 for better convergence
        "patience": 5,  # early stopping on val_macro_f1
        # Paper §4.2: lr=5e-6 for PLMs on ViFactCheck (NOT 2e-5)
        "lr": 5e-6,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "grad_clip": 1.0,
        "label_smoothing": 0.0,
        "seed": 42,
    },
    "mlflow": {
        "experiment_name": "vifactcheck-stage39-text",
    },
    "safety": {
        "smoke_test": False,
        "smoke_batches": 2,
        "auto_install_deps": False,
        "resume_from_checkpoint": None,
        # Set True to re-extract stage39 features even if .h5 files already exist
        "force_rebuild_stage39_features": False,
    },
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")
print(f"Checkpoint root:     {CONFIG['paths']['checkpoint_root']}")
print(f"Stage39 features:    {CONFIG['paths']['stage39_features_dir']}")

Project root: /Users/haila/My File/projects/fake-new-detection
Data root:    /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis
Checkpoint root: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/training/checkpoints_vifactcheck


In [3]:
# ── DEPENDENCY PREFLIGHT ────────────────────────────────────────────────────
import importlib, sys

_required = {
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
    "mlflow": "mlflow",
}

_missing = []
for mod, pkg in _required.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        _missing.append(pkg)

if _missing:
    if CONFIG["safety"]["auto_install_deps"]:
        import subprocess

        subprocess.check_call([sys.executable, "-m", "pip", "install"] + _missing)
        print(f"Installed: {_missing}")
    else:
        print(f"Missing packages: {_missing}")
        print(
            "Install with:  pip install transformers datasets scikit-learn mlflow seaborn tqdm"
        )
        raise RuntimeError(f"Missing packages: {_missing}.")
else:
    print("All dependencies satisfied.")

All dependencies satisfied.


In [4]:
# ── IMPORTS AND SETUP ───────────────────────────────────────────────────────
import sys, os, gc, json, random, hashlib
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
)
from transformers import AutoTokenizer, AutoModel

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG["paths"]["checkpoint_root"].mkdir(parents=True, exist_ok=True)

print(f"PyTorch      : {torch.__version__}")
import transformers

print(f"Transformers : {transformers.__version__}")


def select_device():
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"Device: cuda ({torch.cuda.get_device_name(0)}, {mem:.1f} GB)")
    elif torch.backends.mps.is_available():
        dev = torch.device("mps")
        print("Device: mps (Apple Silicon). For full training, prefer a CUDA GPU.")
    else:
        dev = torch.device("cpu")
        print("Device: cpu. Use smoke_test=True for local validation.")
    return dev


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    print(f"Seed: {seed}")


DEVICE = select_device()
seed_everything(CONFIG["training"]["seed"])

PyTorch      : 2.12.0
Transformers : 5.10.2
Device: mps (Apple Silicon). For full training, prefer a CUDA GPU.
Seed: 42


## Step 1: Load ViFactCheck Dataset

Loads from HuggingFace `tranthaihoa/vifactcheck` with auto-detected text/label fields,
or falls back to `data/json/labeled_nei_as_real/` JSON files.
`nei_as_real` strategy: Supported(0)→0, NEI(2)→0, Refuted(1)→1.


In [5]:
def _apply_label_strategy(raw_label, strategy):
    if raw_label is None:
        return None
    raw_label = int(raw_label)
    if strategy == "nei_as_real":
        if raw_label == 1:
            return 1  # Refute → Fake
        elif raw_label in (0, 2):
            return 0  # Support / NEI → Real
        return None
    elif strategy == "binary_exclude_nei":
        if raw_label == 0:
            return 0
        elif raw_label == 1:
            return 1
        return None
    raise ValueError(f"Unknown strategy: {strategy}")


_segmenter = None


def _segment_text(text, use_segmentation):
    """
    Optional Vietnamese word segmentation for PhoBERT.
    Paper §4.2: PhoBERT requires VnCoreNLP segmentation; underthesea is a
    drop-in Python alternative (pip install underthesea).
    Output: space-joined word tokens, e.g. 'Phó_Thủ_tướng Trần_Hồng_Hà ...'
    """
    if not use_segmentation or not text:
        return text
    global _segmenter
    if _segmenter is None:
        try:
            from underthesea import word_tokenize

            _segmenter = word_tokenize
            print("  Word segmentation: underthesea loaded.")
        except ImportError:
            print("  ⚠  underthesea not installed — skipping word segmentation.")
            print("     Install with: pip install underthesea")
            _segmenter = lambda t, format=None: t
    return _segmenter(text, format="text")


def _detect_hf_fields(sample):
    main_cands = [
        "Claim",
        "claim",
        "Statement",
        "statement",
        "text",
        "title",
        "headline",
        "article",
    ]
    ctx_cands = [
        "Evidence",
        "evidence",
        "Context",
        "context",
        "summary",
        "article_text",
        "body",
    ]
    lbl_cands = ["labels", "label", "verdict", "Verdict", "Label", "gold_label"]
    main = next((f for f in main_cands if f in sample), None)
    ctx = next((f for f in ctx_cands if f in sample), None)
    lbl = next((f for f in lbl_cands if f in sample), None)
    return main, ctx, lbl


def _find_hf_split(candidates, available):
    return next((c for c in candidates if c in available), None)


def _build_split_map(available):
    train_cands = ["train", "train_data", "training", "Train"]
    val_cands = ["validation", "val", "dev", "valid", "validation_data", "Val", "Dev"]
    test_cands = ["test", "test_data", "testing", "Test"]

    train_key = _find_hf_split(train_cands, available)
    val_key = _find_hf_split(val_cands, available)
    test_key = _find_hf_split(test_cands, available)

    if len(available) == 1:
        only = available[0]
        print(f"  ⚠  Single split '{only}' — using it for train/dev/test.")
        return {"train": only, "dev": only, "test": only}

    split_map = {}
    if train_key:
        split_map["train"] = train_key
    elif available:
        split_map["train"] = available[0]
        print(f"  ⚠  No standard train split; using '{available[0]}' as train.")
    if val_key:
        split_map["dev"] = val_key
    if test_key:
        split_map["test"] = test_key
    return split_map


def load_from_huggingface(config):
    from datasets import load_dataset

    hf_name = config["data"]["hf_dataset"]
    strategy = config["data"]["label_strategy"]
    nc = config["data"]["num_classes"]
    use_seg = config["data"]["word_segment"]

    print(f"Loading '{hf_name}' from HuggingFace...")
    raw = load_dataset(hf_name)

    available = list(raw.keys())
    print(f"  Available HF splits : {available}")
    sample = dict(raw[available[0]][0])
    print(f"  Columns             : {list(sample.keys())}")
    print(f"  First sample        : { {k: str(v)[:60] for k, v in sample.items()} }")

    main_field = config["data"]["hf_text_field"] or _detect_hf_fields(sample)[0]
    ctx_field = config["data"]["hf_context_field"] or _detect_hf_fields(sample)[1]
    label_field = config["data"]["hf_label_field"] or _detect_hf_fields(sample)[2]

    print(f"\n  text_field    = {main_field!r}  (Statement = claim to verify)")
    print(
        f"  context_field = {ctx_field!r}  (Evidence = gold evidence, best mode per §4.3)"
    )
    print(f"  label_field   = {label_field!r}")
    print(f"  word_segment  = {use_seg}")

    if main_field is None:
        raise ValueError(
            f"Cannot auto-detect text field from columns {list(sample.keys())}.\n"
            f"Set CONFIG['data']['hf_text_field'] = '<column name>' manually."
        )
    if label_field is None:
        raise ValueError(
            f"Cannot auto-detect label field from columns {list(sample.keys())}.\n"
            f"Set CONFIG['data']['hf_label_field'] = '<column name>' manually."
        )

    split_map = _build_split_map(available)
    print(f"  Split map           : {split_map}")

    all_examples = {}
    for our_split, hf_split in split_map.items():
        ds = raw[hf_split]
        examples, skipped = [], 0
        for item in ds:
            mapped = _apply_label_strategy(item.get(label_field), strategy)
            if mapped is None:
                skipped += 1
                continue
            text_a = _segment_text(str(item.get(main_field) or "").strip(), use_seg)
            text_b_raw = item.get(ctx_field) if ctx_field else None
            if isinstance(text_b_raw, list):
                text_b_raw = " ".join(str(x) for x in text_b_raw if x)
            text_b_str = str(text_b_raw or "").strip()
            text_b = _segment_text(text_b_str, use_seg) if text_b_str else None
            if not text_a:
                skipped += 1
                continue
            examples.append({"text": text_a, "text_b": text_b, "label": mapped})
        hist = [sum(1 for e in examples if e["label"] == i) for i in range(nc)]
        print(
            f"  {our_split:5s}: {len(examples):5d} examples (skipped {skipped:4d}) | {hist}"
        )
        all_examples[our_split] = examples
    return all_examples


def load_from_labeled_json(config):
    json_dir = config["paths"]["labeled_json_dir"]
    variant = config["data"]["label_variant"]
    strategy = config["data"]["label_strategy"]
    nc = config["data"]["num_classes"]
    use_seg = config["data"]["word_segment"]

    sub = {
        "nei_as_real": "labeled_nei_as_real",
        "three_class": "labeled_three_class",
    }.get(variant, "")
    split_dir = json_dir / sub if sub else json_dir

    all_examples = {}
    for split in ["train", "dev", "test"]:
        json_path = split_dir / f"news_data_vifactcheck_{split}_labeled.json"
        if not json_path.exists():
            print(f"  ⚠  Not found: {json_path}. Skipping.")
            continue
        with open(json_path, "r", encoding="utf-8") as f:
            articles = json.load(f)
        examples, skipped = [], 0
        for art in articles:
            mapped = _apply_label_strategy(art.get("label"), strategy)
            if mapped is None:
                skipped += 1
                continue
            title = str(art.get("title") or "").strip()
            captions = " ".join(
                str(img.get("caption") or "").strip()
                for img in art.get("images", [])
                if img.get("caption")
            )
            raw_text = (title + " " + captions).strip() or title
            text_a = _segment_text(raw_text, use_seg)
            if not text_a:
                skipped += 1
                continue
            examples.append({"text": text_a, "text_b": None, "label": mapped})
        hist = [sum(1 for e in examples if e["label"] == i) for i in range(nc)]
        print(
            f"  {split:5s}: {len(examples):5d} examples (skipped {skipped:4d}) | {hist}"
        )
        all_examples[split] = examples
    return all_examples


# ── Load ───────────────────────────────────────────────────────────────────
if CONFIG["data"]["source"] == "huggingface":
    try:
        raw_examples = load_from_huggingface(CONFIG)
        DATA_SOURCE_USED = "huggingface"
    except Exception as _hf_err:
        print(f"\n⚠  HuggingFace failed: {_hf_err}\n   Falling back to labeled JSON...")
        raw_examples = load_from_labeled_json(CONFIG)
        DATA_SOURCE_USED = "labeled_json"
else:
    raw_examples = load_from_labeled_json(CONFIG)
    DATA_SOURCE_USED = "labeled_json"

print(f"\nData source used : {DATA_SOURCE_USED}")
for _s, _ex in raw_examples.items():
    print(f"  {_s}: {len(_ex)} examples")
if "train" not in raw_examples:
    print("\n⚠  WARNING: no 'train' split loaded.")
    print("   Available:", list(raw_examples.keys()))
    print(
        "   If using HF, set CONFIG['data']['hf_text_field'] to the correct column name."
    )

Loading 'tranthaihoa/vifactcheck' from HuggingFace...


  Available HF splits : ['train', 'test', 'dev']
  Columns             : ['Unnamed: 0', 'index', 'Statement', 'Context', 'annotation_id', 'Topic', 'Author', 'Url', 'labels', 'Evidence']
  First sample        : {'Unnamed: 0': '0', 'index': '3049', 'Statement': 'Phó Thủ tướng Trần Hồng Hà thay mặt Chính phủ, Thủ tướng Chí', 'Context': '(Chinhphu.vn) - Đây là mong muốn, gửi gắm của Phó Thủ tướng ', 'annotation_id': '18933775', 'Topic': 'Chính trị', 'Author': 'Chính Phủ', 'Url': 'https://baochinhphu.vn/lan-toa-nhung-gia-tri-van-hoa-tot-dep', 'labels': '0', 'Evidence': 'Thay mặt Chính phủ, Thủ tướng Chính phủ, Phó Thủ tướng Trần '}

  text_field    = 'Statement'  (Statement = claim to verify)
  context_field = 'Evidence'  (Evidence = gold evidence, best mode per §4.3)
  label_field   = 'labels'
  word_segment  = False
  Split map           : {'train': 'train', 'dev': 'dev', 'test': 'test'}
  train:  5062 examples (skipped    0) | [3404, 1658]
  dev  :   723 examples (skipped    0) | [479, 2

## Step 2: Tokenize and Build DataLoaders

Tokenizes using `phobert-base-v2`'s tokenizer with sentence-pair encoding when a context field is present,
otherwise single-sentence. Wraps splits into `ViFactCheckDataset` and standard `DataLoader`s.


In [6]:
class ViFactCheckDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(
            ex["text"],
            ex["text_b"],
            max_length=self.max_length,
            padding="max_length",
            # "only_second": keep full Statement (claim), truncate Evidence if too long
            # avoids the 'longest_first' warning and is semantically correct
            truncation="only_second" if ex["text_b"] is not None else True,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(ex["label"], dtype=torch.long),
        }


if "train" not in raw_examples:
    raise KeyError(
        f"No 'train' split in loaded data. Got splits: {list(raw_examples.keys())}.\n"
        f"Check the Step 1 output for split-detection messages.\n"
        f"You can also switch to the labeled JSON fallback by setting:\n"
        f"  CONFIG['data']['source'] = 'labeled_json'\nthen re-running from Step 1."
    )

print(f"Loading tokenizer: {CONFIG['model']['backbone']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model"]["backbone"])
print("Tokenizer loaded.")

_max_len = CONFIG["data"]["max_length"]
_bs = CONFIG["training"]["batch_size"]
_smoke = CONFIG["safety"]["smoke_test"]
_smoke_n = CONFIG["safety"]["smoke_batches"]

datasets_map = {
    split: ViFactCheckDataset(exs, tokenizer, _max_len)
    for split, exs in raw_examples.items()
}


def _make_loader(ds, shuffle, batch_size):
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)


loaders = {
    "train": _make_loader(datasets_map["train"], shuffle=True, batch_size=_bs),
    "dev": _make_loader(
        datasets_map.get("dev", datasets_map["train"]), shuffle=False, batch_size=_bs
    ),
    "test": _make_loader(
        datasets_map.get("test", datasets_map["train"]), shuffle=False, batch_size=_bs
    ),
}

# Smoke-test wrapper
if _smoke:
    from itertools import islice

    class _SmokeLoader:
        def __init__(self, loader, n):
            self._l = loader
            self._n = n

        def __iter__(self):
            return islice(self._l, self._n)

        def __len__(self):
            return min(self._n, len(self._l))

    loaders = {k: _SmokeLoader(v, _smoke_n) for k, v in loaders.items()}
    print(f"SMOKE TEST: {_smoke_n} batches per split")

# Sanity check batch
_b = next(iter(loaders["train"]))
print(
    f"\nBatch shapes — input_ids: {tuple(_b['input_ids'].shape)}  "
    f"attention_mask: {tuple(_b['attention_mask'].shape)}  "
    f"label: {tuple(_b['label'].shape)}"
)
for split in ["train", "dev", "test"]:
    _ds = datasets_map.get(split)
    if _ds:
        hist = [
            sum(1 for e in _ds.examples if e["label"] == i)
            for i in range(CONFIG["data"]["num_classes"])
        ]
        print(f"  {split:5s}: {len(_ds):5d} samples | label dist {hist}")

Loading tokenizer: vinai/phobert-base-v2
Tokenizer loaded.

Batch shapes — input_ids: (16, 256)  attention_mask: (16, 256)  label: (16,)
  train:  5062 samples | label dist [3404, 1658]
  dev  :   723 samples | label dist [479, 244]
  test :  1447 samples | label dist [979, 468]


## Step 3: Build PhoBERT Classifier

`PhoBERTClassifier` wraps `phobert-base-v2` with a dropout + linear head over the `[CLS]` token.
`get_cls_features()` exposes the raw 768-dim `[CLS]` vector for Stage 4 integration.


In [7]:
class PhoBERTClassifier(nn.Module):
    """PhoBERT fine-tuned text classifier for Vietnamese fake-news detection."""

    def __init__(self, backbone_name, num_classes, dropout):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(backbone_name)
        hidden_size = self.backbone.config.hidden_size  # 768 for phobert-base
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes),
        )
        self.hidden_size = hidden_size

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]  # [B, 768]
        return self.classifier(cls)  # [B, num_classes]

    def get_cls_features(self, input_ids, attention_mask):
        """Extract frozen [CLS] embeddings (768-dim) for Stage 4 text encoder."""
        with torch.no_grad():
            out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state[:, 0, :]  # [B, 768]


print(f"Building model: {CONFIG['model']['backbone']}")
model = PhoBERTClassifier(
    backbone_name=CONFIG["model"]["backbone"],
    num_classes=CONFIG["model"]["num_classes"],
    dropout=CONFIG["model"]["dropout"],
).to(DEVICE)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters — total: {total:,}  trainable: {trainable:,}")
print(
    f"Hidden size: {model.hidden_size}  →  classifier → {CONFIG['model']['num_classes']} classes"
)

# One-forward sanity check
model.eval()
with torch.no_grad():
    _ids = _b["input_ids"][:2].to(DEVICE)
    _mask = _b["attention_mask"][:2].to(DEVICE)
    _logit = model(_ids, _mask)
print(
    f"Sanity forward output shape: {tuple(_logit.shape)}  (expected [2, {CONFIG['model']['num_classes']}])"
)

Building model: vinai/phobert-base-v2


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters — total: 134,999,810  trainable: 134,999,810
Hidden size: 768  →  classifier → 2 classes
Sanity forward output shape: (2, 2)  (expected [2, 2])


## Step 4: Loss, Optimizer, and Scheduler

AdamW with linear warmup + linear decay. Class weights derived from training label frequencies
to handle class imbalance.


In [8]:
train_labels = np.array([ex["label"] for ex in raw_examples["train"]], dtype=np.int64)
nc = CONFIG["data"]["num_classes"]
counts = np.bincount(train_labels, minlength=nc).astype(np.float64)
class_weights = torch.tensor(len(train_labels) / (nc * counts), dtype=torch.float32).to(
    DEVICE
)
print(f"Class counts : {counts.astype(int).tolist()}")
print(f"Class weights: {class_weights.cpu().tolist()}")

loss_fn = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=CONFIG["training"]["label_smoothing"],
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["training"]["lr"],
    weight_decay=CONFIG["training"]["weight_decay"],
)

total_steps = CONFIG["training"]["max_epochs"] * len(loaders["train"])
warmup_steps = int(total_steps * CONFIG["training"]["warmup_ratio"])


def _lr_lambda(step):
    if step < warmup_steps:
        return float(step) / max(1, warmup_steps)
    progress = float(step - warmup_steps) / max(1, total_steps - warmup_steps)
    return max(0.0, 1.0 - progress)


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)

print(
    f"\nOptimizer : AdamW  lr={CONFIG['training']['lr']}  wd={CONFIG['training']['weight_decay']}"
)
print(
    f"Scheduler : linear warmup ({warmup_steps} steps) → linear decay ({total_steps} total)"
)

Class counts : [3404, 1658]
Class weights: [0.7435370087623596, 1.5265380144119263]

Optimizer : AdamW  lr=5e-06  wd=0.01
Scheduler : linear warmup (317 steps) → linear decay (3170 total)


## Step 5: MLflow + Run Directory Setup


In [9]:
import mlflow

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"vifactcheck_stage39_{timestamp}"

run_dir = CONFIG["paths"]["checkpoint_root"] / run_name
artifact_dir = run_dir / "artifacts"
run_dir.mkdir(parents=True, exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
print(f"Run dir: {run_dir}")

mlflow_enabled = False
mlflow_run_id = None
try:
    mlflow.set_tracking_uri(CONFIG["paths"]["mlflow_dir"].as_uri())
    mlflow.set_experiment(CONFIG["mlflow"]["experiment_name"])
    _mlflow_run = mlflow.start_run(run_name=run_name)
    mlflow_run_id = _mlflow_run.info.run_id
    mlflow.log_params(
        {
            "backbone": CONFIG["model"]["backbone"],
            "batch_size": CONFIG["training"]["batch_size"],
            "max_epochs": CONFIG["training"]["max_epochs"],
            "lr": CONFIG["training"]["lr"],
            "patience": CONFIG["training"]["patience"],
            "max_length": CONFIG["data"]["max_length"],
            "data_source": DATA_SOURCE_USED,
            "label_strategy": CONFIG["data"]["label_strategy"],
            "smoke_test": CONFIG["safety"]["smoke_test"],
            "seed": CONFIG["training"]["seed"],
        }
    )
    mlflow_enabled = True
    print(f"MLflow run: {mlflow_run_id}")
except Exception as _mlf_err:
    print(f"MLflow disabled; continuing with local artifacts only ({_mlf_err})")

Run dir: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/training/checkpoints_vifactcheck/vifactcheck_stage39_20260615_234341
MLflow disabled; continuing with local artifacts only (The filesystem tracking backend (e.g., './mlruns') is in maintenance mode and will not receive further updates. Please migrate to a database backend (e.g., 'sqlite:///mlflow.db') to access the latest MLflow features. The `mlflow migrate-filestore` tool migrates your existing data losslessly. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance. If the filesystem backend is required for your workflow, set `MLFLOW_ALLOW_FILE_STORE=true` to opt out of this exception.)


## Step 6: Checkpoint Helpers


In [ ]:
def _cfg_jsonable(cfg):
    if isinstance(cfg, dict):
        return {k: _cfg_jsonable(v) for k, v in cfg.items()}
    if isinstance(cfg, Path):
        return str(cfg)
    if isinstance(cfg, (list, tuple)):
        return [_cfg_jsonable(v) for v in cfg]
    return cfg


def save_checkpoint(path, model, epoch, metrics, config, history, mlflow_run_id=None):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "config": _cfg_jsonable(config),
            "epoch": epoch,
            "metrics": metrics,
            "training_history": history,
            "backbone": config["model"]["backbone"],
            "hidden_size": model.hidden_size,
            "num_classes": config["model"]["num_classes"],
            "freeze_for_stage4": True,
            "mlflow_run_id": mlflow_run_id,
        },
        path,
    )


def write_checkpoint_manifest(
    manifest_path,
    best_ckpt_path,
    best_epoch,
    best_metrics,
    config,
    mlflow_enabled,
    mlflow_run_id,
):
    stage39_feats_dir = config["paths"]["stage39_features_dir"]
    manifest = {
        "best_checkpoint_path": str(best_ckpt_path),
        "tokenizer_path": str(best_ckpt_path.parent / "tokenizer"),
        "best_epoch": best_epoch,
        "selection_metric": "val_macro_f1",
        "best_metrics": best_metrics,
        "backbone": config["model"]["backbone"],
        "hidden_size": 768,
        "num_classes": config["model"]["num_classes"],
        "freeze_for_stage4": True,
        "data_source": DATA_SOURCE_USED,
        "label_strategy": config["data"]["label_strategy"],
        "mlflow_enabled": mlflow_enabled,
        "mlflow_run_id": mlflow_run_id,
        # ── Stage 4 integration ───────────────────────────────────────────────
        # How Stage 4 loads Stage 3.9 for Ablation E (phobert_gated):
        #   1. Load best_model.pth into PhoBERTClassifier(backbone, num_classes, dropout)
        #   2. Freeze all params; call get_cls_features(input_ids, attention_mask) → [B, 768]
        #   3. Or use pre-extracted features from stage39_features_dir (faster for training)
        "stage4_integration": {
            "phobert_feature_dim": 768,
            "stage4_text_dim_for_phobert": 768,
            "stage39_features_dir": str(stage39_feats_dir),
            "stage39_train_h5": str(stage39_feats_dir / "stage39_train.h5"),
            "stage39_dev_h5": str(stage39_feats_dir / "stage39_dev.h5"),
            "stage39_test_h5": str(stage39_feats_dir / "stage39_test.h5"),
            "ablation_name": "phobert_gated",
            "description": (
                "Replace COOLANT text_aligned_clip (64-dim) with Stage3.9 PhoBERT CLS (768-dim). "
                "Pair with COOLANT image_aligned_clip (64-dim). "
                "GatedFusionHead(text_dim=768, image_dim=64) → 2 classes."
            ),
            "usage": (
                "from src.models.phobert_classifier import PhoBERTClassifier; "
                "ckpt = torch.load(best_checkpoint_path); "
                "model = PhoBERTClassifier(backbone, num_classes, dropout); "
                "model.load_state_dict(ckpt['model_state_dict']); model.eval(); "
                "feats = model.get_cls_features(input_ids, attention_mask)  # [B, 768]"
            ),
        },
    }
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"Manifest written: {manifest_path}")


print("Checkpoint helpers defined.")

Checkpoint helpers defined.


## Step 7: Training and Evaluation Functions


In [11]:
def compute_metrics(y_true, y_pred, prefix, nc):
    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    per_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    metrics = {
        f"{prefix}_accuracy": round(acc, 4),
        f"{prefix}_macro_f1": round(macro_f1, 4),
    }
    for i in range(nc):
        metrics[f"{prefix}_f1_class{i}"] = round(
            float(per_f1[i]) if i < len(per_f1) else 0.0, 4
        )
    return metrics


def run_epoch(model, loader, loss_fn, optimizer, scheduler, device, config, train):
    model.train() if train else model.eval()
    total_loss, n_batches = 0.0, 0
    y_true_all, y_pred_all = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        pbar = tqdm(loader, desc="  train" if train else "  eval", leave=False)
        for batch in pbar:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            lbl = batch["label"].to(device)

            logits = model(ids, mask)
            loss = loss_fn(logits, lbl)

            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss: {loss.item()}")

            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), config["training"]["grad_clip"]
                )
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            total_loss += loss.item()
            n_batches += 1
            preds = logits.argmax(dim=-1).cpu().numpy()
            y_pred_all.extend(preds.tolist())
            y_true_all.extend(lbl.cpu().numpy().tolist())
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    mean_loss = total_loss / max(1, n_batches)
    return mean_loss, np.array(y_true_all), np.array(y_pred_all)


print("Training and evaluation functions defined.")

Training and evaluation functions defined.


## Step 8: Run Training

Epoch loop with early stopping on `val_macro_f1`. Best checkpoint saved to `run_dir/best_model.pth`.
Tokenizer also saved alongside for easy Stage 4 loading.


In [12]:
patience = CONFIG["training"]["patience"]
max_epochs = CONFIG["training"]["max_epochs"]
nc = CONFIG["data"]["num_classes"]
best_val_f1 = -1.0
best_epoch = -1
no_improve = 0
history = []
best_ckpt_path = run_dir / "best_model.pth"

# Optional resume
_resume = CONFIG["safety"]["resume_from_checkpoint"]
start_epoch = 0
if _resume and Path(_resume).exists():
    _ckpt = torch.load(_resume, map_location=DEVICE)
    model.load_state_dict(_ckpt["model_state_dict"])
    start_epoch = _ckpt.get("epoch", 0) + 1
    print(f"Resumed from epoch {_ckpt.get('epoch', '?')}: {_resume}")

print(f"Training for up to {max_epochs} epochs (patience={patience})...")
print(f"Best checkpoint → {best_ckpt_path}\n")

try:
    for epoch in range(start_epoch, max_epochs):
        # ── Train ──────────────────────────────────────────────────────────
        train_loss, y_tr, yp_tr = run_epoch(
            model,
            loaders["train"],
            loss_fn,
            optimizer,
            scheduler,
            DEVICE,
            CONFIG,
            train=True,
        )
        train_metrics = compute_metrics(y_tr, yp_tr, "train", nc)
        train_metrics["train_loss"] = round(train_loss, 4)

        # ── Val ────────────────────────────────────────────────────────────
        val_loss, y_vl, yp_vl = run_epoch(
            model,
            loaders["dev"],
            loss_fn,
            None,
            None,
            DEVICE,
            CONFIG,
            train=False,
        )
        val_metrics = compute_metrics(y_vl, yp_vl, "val", nc)
        val_metrics["val_loss"] = round(val_loss, 4)
        val_metrics["val_confusion_matrix"] = confusion_matrix(
            y_vl, yp_vl, labels=list(range(nc))
        ).tolist()

        lr_now = scheduler.get_last_lr()[0]
        epoch_record = {
            "epoch": epoch,
            "lr": round(lr_now, 8),
            **train_metrics,
            **val_metrics,
        }
        history.append(epoch_record)

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f}  train_acc={train_metrics['train_accuracy']:.4f}  "
            f"train_f1={train_metrics['train_macro_f1']:.4f} | "
            f"val_loss={val_loss:.4f}  val_acc={val_metrics['val_accuracy']:.4f}  "
            f"val_f1={val_metrics['val_macro_f1']:.4f}"
        )

        if mlflow_enabled:
            mlflow.log_metrics(
                {
                    "train_loss": train_loss,
                    "train_acc": train_metrics["train_accuracy"],
                    "train_macro_f1": train_metrics["train_macro_f1"],
                    "val_loss": val_loss,
                    "val_acc": val_metrics["val_accuracy"],
                    "val_macro_f1": val_metrics["val_macro_f1"],
                    "lr": lr_now,
                },
                step=epoch,
            )

        # ── Checkpoint + Early Stopping ────────────────────────────────────
        val_f1 = val_metrics["val_macro_f1"]
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch
            no_improve = 0
            save_checkpoint(
                best_ckpt_path,
                model,
                epoch,
                {**train_metrics, **val_metrics},
                CONFIG,
                history,
                mlflow_run_id,
            )
            print(f"  ✅ New best val_macro_f1={best_val_f1:.4f} → saved checkpoint")
        else:
            no_improve += 1
            print(f"  No improvement ({no_improve}/{patience})")
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

        # Periodic artifact save
        with open(artifact_dir / "training_history.json", "w") as f:
            json.dump(history, f, indent=2)

        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

except KeyboardInterrupt:
    print("Training interrupted by user.")

print(f"\nTraining complete. Best epoch: {best_epoch}  val_macro_f1: {best_val_f1:.4f}")

# Save tokenizer alongside model for easy Stage 4 loading
tokenizer_dir = run_dir / "tokenizer"
tokenizer.save_pretrained(str(tokenizer_dir))
print(f"Tokenizer saved: {tokenizer_dir}")

with open(artifact_dir / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)
print(f"Training history saved: {artifact_dir / 'training_history.json'}")

if mlflow_enabled:
    try:
        mlflow.end_run()
    except Exception:
        pass

Training for up to 10 epochs (patience=3)...
Best checkpoint → /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/training/checkpoints_vifactcheck/vifactcheck_stage39_20260615_234341/best_model.pth



  train:   0%|          | 0/317 [00:00<?, ?it/s]

Training interrupted by user.

Training complete. Best epoch: -1  val_macro_f1: -1.0000
Tokenizer saved: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/training/checkpoints_vifactcheck/vifactcheck_stage39_20260615_234341/tokenizer
Training history saved: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/training/checkpoints_vifactcheck/vifactcheck_stage39_20260615_234341/artifacts/training_history.json


## Step 9: Final Test Evaluation

Loads the best checkpoint, runs inference on the held-out test split, and produces
accuracy / macro-F1 / per-class F1 + confusion matrix. Results saved to `artifacts/test_report.json`.


In [ ]:
# ── Load best checkpoint for test evaluation ──────────────────────────────
if not best_ckpt_path.exists():
    print("⚠  best_model.pth not found — skipping test evaluation.")
else:
    _ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
    model.load_state_dict(_ckpt["model_state_dict"])
    print(f"Loaded best checkpoint (epoch {_ckpt['epoch']}) for test evaluation.")

    # ── Test eval ─────────────────────────────────────────────────────────
    test_loss, y_te, yp_te = run_epoch(
        model, loaders["test"], loss_fn, None, None, DEVICE, CONFIG, train=False
    )
    test_metrics = compute_metrics(y_te, yp_te, "test", nc)
    test_metrics["test_loss"] = round(test_loss, 4)

    cm_test = confusion_matrix(y_te, yp_te, labels=list(range(nc)))

    print(f"\n{'='*55}")
    print(f"TEST RESULTS  (best epoch: {best_epoch})")
    print(f"{'='*55}")
    for k, v in test_metrics.items():
        if k != "test_loss":
            print(f"  {k:<30s}: {v}")
    print(f"  {'test_loss':<30s}: {test_metrics['test_loss']}")
    print(f"  Confusion matrix (rows=true, cols=pred):\n{cm_test}")

    # ── Save test report ───────────────────────────────────────────────────
    test_report = {
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_f1,
        **test_metrics,
        "confusion_matrix": cm_test.tolist(),
        "label_names": {0: "Real", 1: "Fake"},
    }
    with open(artifact_dir / "test_report.json", "w") as f:
        json.dump(test_report, f, indent=2)
    print(f"\nTest report saved: {artifact_dir / 'test_report.json'}")

## Step 10: Visualize Training Curves and Confusion Matrix


In [ ]:
if not history:
    print("No training history to plot.")
else:
    sns.set_theme(style="whitegrid", palette="muted")
    hist_df = pd.DataFrame(history)
    epochs = hist_df["epoch"]
    best_ep = best_epoch

    # ── Training curves ────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    def _vline(ax):
        ax.axvline(
            best_ep,
            color="red",
            linestyle="--",
            linewidth=1.2,
            label=f"best ({best_ep})",
        )

    axes[0].plot(epochs, hist_df["train_loss"], label="train", marker="o", markersize=3)
    axes[0].plot(epochs, hist_df["val_loss"], label="val", marker="o", markersize=3)
    _vline(axes[0])
    axes[0].set_title("Loss")
    axes[0].legend()

    axes[1].plot(
        epochs, hist_df["train_accuracy"], label="train", marker="o", markersize=3
    )
    axes[1].plot(epochs, hist_df["val_accuracy"], label="val", marker="o", markersize=3)
    _vline(axes[1])
    axes[1].set_title("Accuracy")
    axes[1].legend()

    axes[2].plot(
        epochs, hist_df["train_macro_f1"], label="train", marker="o", markersize=3
    )
    axes[2].plot(epochs, hist_df["val_macro_f1"], label="val", marker="o", markersize=3)
    _vline(axes[2])
    axes[2].set_title("Macro-F1")
    axes[2].legend()

    for ax in axes:
        ax.set_xlabel("Epoch")
    plt.suptitle(
        "ViFactCheck Stage 3.9 — Training Curves", fontsize=13, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(artifact_dir / "training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ── Confusion matrix ───────────────────────────────────────────────────
    if best_ckpt_path.exists():
        labels = ["Real", "Fake"]
        cm_norm = cm_test.astype(float) / cm_test.sum(axis=1, keepdims=True)

        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        for ax, data, fmt, title in zip(
            axes,
            [cm_test, cm_norm],
            ["d", ".2f"],
            ["Counts", "Row-normalized"],
        ):
            sns.heatmap(
                data,
                annot=True,
                fmt=fmt,
                cmap="Blues",
                xticklabels=labels,
                yticklabels=labels,
                ax=ax,
                vmin=0 if fmt == ".2f" else None,
                vmax=1 if fmt == ".2f" else None,
            )
            ax.set_xlabel("Predicted")
            ax.set_ylabel("Actual")
            ax.set_title(f"Confusion Matrix ({title})")

        plt.suptitle(
            f"Test  acc={test_metrics['test_accuracy']:.4f}  "
            f"macro-F1={test_metrics['test_macro_f1']:.4f}",
            fontsize=12,
            fontweight="bold",
        )
        plt.tight_layout()
        plt.savefig(
            artifact_dir / "test_confusion_matrix.png", dpi=150, bbox_inches="tight"
        )
        plt.show()

## Step 11: Write Checkpoint Manifest (Stage 4 Handoff)

Writes `checkpoint_manifest.json` under the run directory. Stage 4 (or any downstream notebook)
can load this to get the fine-tuned PhoBERT path and call `get_cls_features()` for 768-dim text features.


In [ ]:
if best_ckpt_path.exists():
    manifest_path = run_dir / "checkpoint_manifest.json"

    best_metrics_for_manifest = {
        "best_val_macro_f1": best_val_f1,
        "best_epoch": best_epoch,
    }
    if "test_metrics" in dir() and test_metrics:
        best_metrics_for_manifest.update(
            {
                "test_accuracy": test_metrics.get("test_accuracy"),
                "test_macro_f1": test_metrics.get("test_macro_f1"),
                "test_f1_class0": test_metrics.get("test_f1_class0"),
                "test_f1_class1": test_metrics.get("test_f1_class1"),
            }
        )

    write_checkpoint_manifest(
        manifest_path,
        best_ckpt_path,
        best_epoch,
        best_metrics_for_manifest,
        CONFIG,
        mlflow_enabled,
        mlflow_run_id,
    )

    print(f"\n{'='*60}")
    print("Stage 3.9 complete. Outputs:")
    print(f"  Best checkpoint  : {best_ckpt_path}")
    print(f"  Tokenizer        : {run_dir / 'tokenizer'}")
    print(f"  Manifest         : {manifest_path}")
    print(f"  Artifacts        : {artifact_dir}")
    print(f"\nStage 4 handoff:")
    print(f"  PhoBERTClassifier.get_cls_features(input_ids, attention_mask) → [B, 768]")
    print(f"  Manifest path for Stage 4: {manifest_path}")
else:
    print("⚠  No checkpoint found. Training may not have completed successfully.")

## Step 12: Stage 4 Feature Bridge

Extracts per-article PhoBERT `[CLS]` embeddings (768-dim) from local ViFactCheck JSON files
(same articles Stage 4 uses) and saves them to `training/stage39_features/{split}.h5`.

Stage 4 can then use these as **Ablation E** (`phobert_gated`):
`PhoBERT text (768-dim) + COOLANT image (64-dim)` → `GatedFusionHead(text_dim=768, image_dim=64)`

```
stage39_features/
  stage39_train.h5  → text_features [N, 768], labels [N], article_ids [N]
  stage39_dev.h5
  stage39_test.h5
```

> **Requires** a completed Stage 3.9 best checkpoint. Skipped if .h5 files already exist
> (set `CONFIG["safety"]["force_rebuild_stage39_features"] = True` to re-extract).


In [ ]:
import h5py


def _build_stage39_features(
    split,
    json_path,
    tokenizer,
    model,
    device,
    out_path,
    max_length,
    batch_size,
    label_strategy,
    force_rebuild,
):
    if out_path.exists() and not force_rebuild:
        print(
            f"  [cache] {split}: {out_path} exists — skipping (set force_rebuild=True to re-extract)"
        )
        return

    if not json_path.exists():
        print(f"  ⚠  Local JSON not found: {json_path}  — skipping {split}")
        return

    with open(json_path, "r", encoding="utf-8") as f:
        articles = json.load(f)

    # Build (text, label, article_id) per article
    texts, labels_list, art_ids = [], [], []
    for idx, art in enumerate(articles):
        raw_label = art.get("label")
        mapped = _apply_label_strategy(raw_label, label_strategy)
        if mapped is None:
            continue
        title = str(art.get("title") or "").strip()
        content = str(art.get("content") or "").strip()
        text = (title + " " + content[:200]).strip() if content else title
        if not text:
            continue
        texts.append(text)
        labels_list.append(mapped)
        art_ids.append(idx)

    if not texts:
        print(f"  ⚠  No valid articles in {json_path.name} — skipping {split}")
        return

    # Extract CLS features in batches
    model.eval()
    all_cls = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        enc = tokenizer(
            batch_texts,
            max_length=max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        ids = enc["input_ids"].to(device)
        mask = enc["attention_mask"].to(device)
        cls = model.get_cls_features(ids, mask)  # [B, 768]
        all_cls.append(cls.cpu().numpy())

    text_feats = np.concatenate(all_cls, axis=0).astype(np.float32)  # [N, 768]
    labels_arr = np.array(labels_list, dtype=np.int64)
    aids_arr = np.array(art_ids, dtype=np.int64)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with h5py.File(out_path, "w") as f:
        f.create_dataset("text_features", data=text_feats)
        f.create_dataset("labels", data=labels_arr)
        f.create_dataset("article_ids", data=aids_arr)
        f.attrs["n_samples"] = len(labels_arr)
        f.attrs["phobert_dim"] = 768
        f.attrs["stage39_run_name"] = run_name
        f.attrs["label_strategy"] = label_strategy

    hist = np.bincount(labels_arr, minlength=CONFIG["data"]["num_classes"]).tolist()
    print(
        f"  {split:5s}: {len(labels_arr):5d} articles | labels {hist} → {out_path.name}"
    )


# ── Best checkpoint path ─────────────────────────────────────────────────────
best_ckpt_path = run_dir / "best_model.pth"
if not best_ckpt_path.exists():
    print(f"⚠  best_model.pth not found at {best_ckpt_path}")
    print("   Run training (Step 8) to completion before extracting Stage 4 features.")
else:
    # Load best model weights
    _ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
    model.load_state_dict(_ckpt["model_state_dict"])
    model.eval()
    print(
        f"Loaded best checkpoint (epoch {_ckpt.get('epoch', '?')})  →  extracting Stage 4 features..."
    )

    _feat_dir = CONFIG["paths"]["stage39_features_dir"]
    _json_dir = CONFIG["paths"]["labeled_json_dir"]
    _force = CONFIG["safety"]["force_rebuild_stage39_features"]
    _strategy = CONFIG["data"]["label_strategy"]

    for _split in ["train", "dev", "test"]:
        _json_path = _json_dir / f"news_data_vifactcheck_{_split}_labeled.json"
        _out_path = _feat_dir / f"stage39_{_split}.h5"
        _build_stage39_features(
            split=_split,
            json_path=_json_path,
            tokenizer=tokenizer,
            model=model,
            device=DEVICE,
            out_path=_out_path,
            max_length=CONFIG["data"]["max_length"],
            batch_size=CONFIG["training"]["batch_size"],
            label_strategy=_strategy,
            force_rebuild=_force,
        )

    print(f"\nStage 4 features written to: {_feat_dir}")
    print("Stage 4 can now use these for Ablation E (phobert_gated).")
    print(
        "Set CONFIG['paths']['stage39_features_dir'] in notebook 04 to:", str(_feat_dir)
    )